## Setup and Data Loading

In [ ]:
# ============================================================================
# Part 1: Exploratory Data Analysis (EDA)
# Titanic Dataset
# ============================================================================

# ============================================================================
# IMPORTS & CONFIGURATION
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# Set seed for reproducibility
np.random.seed(42)
sns.set_style("whitegrid")

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================

df = pd.read_csv("../titanic3.csv")

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
df.head()


In [ ]:
# quick look at data statistics
df.describe()

In [ ]:
# Separate features and target
X_raw = df.drop(columns=["survived"])
y_raw = df["survived"]

print(f"\nFeatures shape: {X_raw.shape}")
print(f"Target shape: {y_raw.shape}")
print(f"Class distribution: {y_raw.value_counts().to_dict()}")

## Summary Statistics

In [ ]:
# ============================================================================
# 1.1 SUMMARY STATISTICS
# ============================================================================

print("=" * 60)
print("NUMERIC FEATURE SUMMARY")
print("=" * 60)

numeric_cols = X_raw.select_dtypes(include=[np.number]).columns
numeric_summary = X_raw[numeric_cols].describe().T
numeric_summary["missing_count"] = X_raw[numeric_cols].isnull().sum()
numeric_summary["missing_pct"] = (numeric_summary["missing_count"] / len(X_raw)) * 100

numeric_summary = numeric_summary[["mean", "50%", "std", "count", "missing_count", "missing_pct"]]
numeric_summary.columns = ["Mean", "Median", "Std", "Count", "Missing", "Missing %"]
print(numeric_summary.round(2))

print("\n" + "=" * 60)
print("CATEGORICAL FEATURE SUMMARY")
print("=" * 60)

categorical_cols = X_raw.select_dtypes(include=["object"]).columns
for col in categorical_cols:
    print(f"\n{col.upper()}:")
    print(X_raw[col].value_counts())

## Missing Values Analysis

In [ ]:
# ============================================================================
# MISSING VALUES ANALYSIS
# ============================================================================

print("=" * 60)
print("MISSING VALUES SUMMARY")
print("=" * 60)

missing_count = X_raw.isnull().sum()
missing_pct = (missing_count / len(X_raw)) * 100

missing_df = pd.DataFrame({
    "Column": X_raw.columns,
    "Type": X_raw.dtypes.values,
    "Missing Count": missing_count.values,
    "Missing %": missing_pct.values
})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values("Missing %", ascending=False)

print(missing_df.to_string(index=False))

## Drop columns with high missing percentage

In [ ]:
# ============================================================================
# DROP COLUMNS WITH >30% MISSING VALUES
# ============================================================================

cols_to_drop_high_missing = []
for col in X_raw.columns:
    miss_pct = (X_raw[col].isnull().sum() / len(X_raw)) * 100
    if miss_pct > 30:
        cols_to_drop_high_missing.append(col)

print(f"Columns with >30% missing values: {cols_to_drop_high_missing}")

df_clean = X_raw.copy()
df_clean = df_clean.drop(columns=cols_to_drop_high_missing)

print(f"\nShape after dropping high-missing columns: {df_clean.shape}")
print(f"Remaining columns: {list(df_clean.columns)}")

## Drop High-Cardinality Columns (name, ticket)

In [ ]:
# ============================================================================
# DROP HIGH-CARDINALITY COLUMNS (useless for KNN)
# ============================================================================

print("\n" + "=" * 60)
print("COLUMN CARDINALITY CHECK")
print("=" * 60)

for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        nunique = df_clean[col].nunique()
        print(f"{col:15} | {nunique:5} unique values")

# Drop name and ticket (too many unique values)
high_cardinality_cols = ["name", "ticket"]
df_clean = df_clean.drop(columns=high_cardinality_cols)

print(f"\nDropped: {high_cardinality_cols}")
print(f"Shape after dropping high-cardinality columns: {df_clean.shape}")
print(f"Remaining columns: {list(df_clean.columns)}")

## Target Distribution & Class Balance

In [ ]:
# ============================================================================
# 1.2 TARGET DISTRIBUTION AND CLASS BALANCE
# ============================================================================

fig, ax = plt.subplots(figsize=(8, 5))
survived_counts = y_raw.value_counts()
colors = ["#FF6B6B", "#4ECDC4"]
bars = ax.bar(
    ["Did Not Survive", "Survived"],
    survived_counts.values,
    color=colors,
    edgecolor="black",
    linewidth=1.5,
)
ax.set_ylabel("Count", fontsize=12)
ax.set_title("Survival Distribution", fontsize=14, fontweight="bold")
ax.set_ylim(0, 900)

for bar, count in zip(bars, survived_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 10,
        str(count),
        ha="center",
        fontweight="bold",
        fontsize=12,
    )

total = len(y_raw)
for i, (label, count) in enumerate(survived_counts.items()):
    pct = (count / total) * 100
    ax.text(
        i,
        count / 2,
        f"{pct:.1f}%",
        ha="center",
        va="center",
        fontsize=14,
        color="white",
        fontweight="bold",
    )

os.makedirs("../figures", exist_ok=True)
plt.tight_layout()
plt.savefig("../figures/target_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print("=" * 50)
print("CLASS BALANCE ANALYSIS")
print("=" * 50)
print(f"Total passengers: {len(y_raw)}")
print(f"Survived: {survived_counts[1]} ({survived_counts[1] / len(y_raw) * 100:.2f}%)")
print(f"Did not survive: {survived_counts[0]} ({survived_counts[0] / len(y_raw) * 100:.2f}%)")
print("\nThe dataset is mildly imbalanced with ~38% survival rate.")

## Feature vs Target Investigations

In [ ]:
# ============================================================================
# 1.3 THREE FEATURE-VS-TARGET INVESTIGATIONS
# ============================================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Pclass vs Survival
pclass_survival = pd.crosstab(df_clean["pclass"], y_raw, normalize="index") * 100
pclass_survival.plot(
    kind="bar",
    stacked=True,
    ax=axes[0],
    color=["#FF6B6B", "#4ECDC4"],
    edgecolor="black",
)
axes[0].set_title("Survival Rate by Passenger Class", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Passenger Class")
axes[0].set_ylabel("Percentage")
axes[0].set_ylim(0, 100)
axes[0].legend(["Did Not Survive", "Survived"], loc="upper right")
axes[0].tick_params(axis="x", rotation=0)

# Plot 2: Sex vs Survival
sex_survival = pd.crosstab(df_clean["sex"], y_raw, normalize="index") * 100
sex_survival.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    color=["#FF6B6B", "#4ECDC4"],
    edgecolor="black",
)
axes[1].set_title("Survival Rate by Sex", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Sex")
axes[1].set_ylabel("Percentage")
axes[1].set_ylim(0, 100)
axes[1].legend(["Did Not Survive", "Survived"], loc="upper right")
axes[1].tick_params(axis="x", rotation=0)

# Plot 3: Age Distribution by Survival
df_temp = df_clean.copy()
df_temp["survived"] = y_raw
df_temp[df_temp["age"].notna()].boxplot(column="age", by="survived", ax=axes[2])
axes[2].set_title("Age Distribution by Survival", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Survived (0=No, 1=Yes)")
axes[2].set_ylabel("Age")
axes[2].set_xticklabels(["Did Not Survive", "Survived"])

plt.suptitle("")
plt.tight_layout()
plt.savefig("../figures/feature_target_investigations.png", dpi=300, bbox_inches="tight")
plt.show()

print("Observations:")
print("1. PCLASS: Higher class passengers had significantly higher survival rates")
print("2. SEX: Females had much higher survival rate")
print("3. AGE: Survived and not survived individuals had sligly different patterns. For this data, given old age, the individual was likely to not survive.")

## Age Distribution

In [ ]:
# ============================================================================
# AGE DISTRIBUTION VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(
    df_clean["age"].dropna(),
    bins=30,
    edgecolor="black",
    alpha=0.7,
    color="skyblue",
)
axes[0].axvline(
    df_clean["age"].median(),
    color="red",
    linestyle="--",
    label=f"Median: {df_clean['age'].median():.1f}",
)
axes[0].axvline(
    df_clean["age"].mean(),
    color="green",
    linestyle="--",
    label=f"Mean: {df_clean['age'].mean():.1f}",
)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Age Distribution")
axes[0].legend()

df_clean.boxplot(column="age", ax=axes[1])
axes[1].set_title("Age Box Plot (shows outliers)")
axes[1].set_ylabel("Age")

plt.tight_layout()
plt.savefig("../figures/age_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print("Age Statistics:")
print(f"  Mean: {df_clean['age'].mean():.2f}")
print(f"  Median: {df_clean['age'].median():.2f}")
print(f"  Std Dev: {df_clean['age'].std():.2f}")
print(f"  Skewness: {df_clean['age'].skew():.2f}")

## Fare Column Distribution

In [ ]:
# ============================================================================
# RAW FARE DISTRIBUTION VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(
    df_clean["fare"].dropna(),
    bins=50,
    edgecolor="black",
    alpha=0.7,
    color="lightgreen",
)
axes[0].axvline(
    df_clean["fare"].median(),
    color="red",
    linestyle="--",
    label=f"Median: {df_clean['fare'].median():.2f}",
)
axes[0].axvline(
    df_clean["fare"].mean(),
    color="green",
    linestyle="--",
    label=f"Mean: {df_clean['fare'].mean():.2f}",
)
axes[0].set_xlabel("Fare")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Raw Fare Distribution (Highly Skewed)")
axes[0].legend()

df_clean.boxplot(column="fare", ax=axes[1])
axes[1].set_title("Raw Fare Box Plot (extreme outliers visible)")
axes[1].set_ylabel("Fare")

plt.tight_layout()
plt.savefig("../figures/fare_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print("Raw Fare Statistics:")
print(f"  Mean: ${df_clean['fare'].mean():.2f}")
print(f"  Median: ${df_clean['fare'].median():.2f}")
print(f"  Max: ${df_clean['fare'].max():.2f}")
print(f"  Skewness: {df_clean['fare'].skew():.2f}")

## Log Transform Fare Visualization

In [ ]:
# ============================================================================
# LOG TRANSFORM FARE VISUALIZATION
# ============================================================================

df_clean["fare_log"] = np.log1p(df_clean["fare"])

print("=== FARE TRANSFORMATION ===\n")
print(f"Original Fare Skewness: {df_clean['fare'].skew():.2f}")
print(f"Log Transformed Fare Skewness: {df_clean['fare_log'].skew():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(
    df_clean["fare_log"].dropna(),
    bins=50,
    edgecolor="black",
    alpha=0.7,
    color="lightblue",
)
axes[0].set_xlabel("Log(Fare)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Log-Transformed Fare Distribution (EDA)")

df_clean.boxplot(column="fare_log", ax=axes[1])
axes[1].set_title("Log-Transformed Fare Box Plot")
axes[1].set_ylabel("Log(Fare)")

plt.tight_layout()
plt.savefig("../figures/fare_log_transformed.png", dpi=300, bbox_inches="tight")
plt.show()

# Note: We keep fare_log for now. After imputation, we'll drop raw fare and keep fare_log

## Embarked Distribution

In [ ]:
# ============================================================================
# EMBARKED DISTRIBUTION
# ============================================================================

fig, ax = plt.subplots(figsize=(6, 4))

embarked_counts = df_clean["embarked"].value_counts()
colors = ["#FF6B6B", "#4ECDC4", "#95E77E"]
bars = ax.bar(embarked_counts.index, embarked_counts.values, color=colors)
ax.set_xlabel("Embarkation Port")
ax.set_ylabel("Count")
ax.set_title("Embarkation Port Distribution")

for bar, count in zip(bars, embarked_counts.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 5,
        str(count),
        ha="center",
        fontweight="bold",
    )

total_embarked = len(df_clean["embarked"].dropna())
for i, (port, count) in enumerate(embarked_counts.items()):
    pct = (count / total_embarked) * 100
    ax.text(
        i,
        count / 2,
        f"{pct:.1f}%",
        ha="center",
        va="center",
        fontsize=11,
        color="white",
        fontweight="bold",
    )

plt.tight_layout()
plt.savefig("../figures/embarked_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

print("Embarked Value Counts:")
for port, count in embarked_counts.items():
    print(f"  {port}: {count} ({count / total_embarked * 100:.1f}%)")

mode_port = df_clean["embarked"].mode()[0]
print(f"\nMode: '{mode_port}' ({embarked_counts[mode_port] / total_embarked * 100:.1f}% of passengers)")

## Normality Tests

In [ ]:
# ============================================================================
# NORMALITY TESTS
# ============================================================================

print("=" * 60)
print("NORMALITY TESTS")
print("=" * 60)

# Test for Age
age_data = df_clean["age"].dropna()
shapiro_age = stats.shapiro(age_data)
ks_age = stats.kstest(age_data, "norm", args=(age_data.mean(), age_data.std()))

print("\nAge column normality tests:")
print(f"    Shapiro-Wilk test: statistic={shapiro_age.statistic:.4f}, p-value={shapiro_age.pvalue:.4e}")
print(f"    Kolmogorov-Smirnov test: statistic={ks_age.statistic:.4f}, p-value={ks_age.pvalue:.4e}")

if shapiro_age.pvalue < 0.05:
    print("Age is not normally distributed")
else:
    print("Age is normally distributed")

# Test for Raw Fare
fare_data = df_clean["fare"].dropna()
shapiro_fare = stats.shapiro(fare_data)
ks_fare = stats.kstest(fare_data, "norm", args=(fare_data.mean(), fare_data.std()))

print("\nRaw Fare column Normality Tests:")
print(f"    Shapiro-Wilk test: statistic={shapiro_fare.statistic:.4f}, p-value={shapiro_fare.pvalue:.4e}")
print(f"    Kolmogorov-Smirnov test: statistic={ks_fare.statistic:.4f}, p-value={ks_fare.pvalue:.4e}")

if shapiro_fare.pvalue < 0.05:
    print("Fare is not normally distributed")
else:
    print("Fare is normally distributed")

## Missing Data Imputation

In [ ]:
# ============================================================================
# 1.4 MISSING DATA STRATEGY
# ============================================================================

print("=" * 60)
print("MISSING DATA STRATEGY")
print("=" * 60)

# Impute categorical Embarked with Mode
mode_embarked = df_clean["embarked"].mode()[0]
df_clean["embarked"] = df_clean["embarked"].fillna(mode_embarked)
print(f"Embarked: Imputed with mode '{mode_embarked}'")

# Impute Raw Fare with Median (before log transform)
median_fare = df_clean["fare"].median()
df_clean["fare"] = df_clean["fare"].fillna(median_fare)
print(f"Raw Fare: Imputed with median {median_fare:.2f}")

# Impute Age with Group Median (by pclass and sex)
age_medians = df_clean.groupby(["pclass", "sex"])["age"].transform("median")
df_clean["age"] = df_clean["age"].fillna(age_medians)
print("Age: Imputed with group median (pclass + sex)")

# Now apply log transform to fare (after imputation)
df_clean["fare_log"] = np.log1p(df_clean["fare"])

# Drop the original raw fare column (keep only fare_log)
df_clean = df_clean.drop(columns=["fare"])

print(f"\nfare_log created and raw fare dropped")
print(f"fare_log skewness after imputation: {df_clean['fare_log'].skew():.2f}")

# Verify no missing values remain
remaining_missing = df_clean.isnull().sum().sum()
print(f"\nRemaining missing values: {remaining_missing}")

# Display missing data strategy table
print("\n" + "-" * 60)
print("Missing Data Strategy Summary:")
print("-" * 60)
strategy_df = pd.DataFrame({
    "Feature": ["cabin", "boat", "body", "home.dest", "name", "ticket", "age", "fare", "embarked"],
    "Missing %": [78.5, 62.8, 91.2, 43.0, 0, 0, 19.96, 0.1, 0],
    "Strategy": ["Dropped", "Dropped", "Dropped", "Dropped", "Dropped", "Dropped", "Group Median", "Median + Log Transform", "Mode"],
    "Justification": [">30% threshold", ">30% threshold", ">30% threshold", ">30% threshold",
                      "High cardinality (1047 unique)", "High cardinality (789 unique)",
                      "Non-normal distribution", "Non-normal distribution, log reduces skewness", "Categorical, mode preserves distribution"]
})
print(strategy_df.to_string(index=False))

## Encoding & Final Feature Matrix

In [ ]:
# ============================================================================
# 1.5 ENCODING AND FINAL FEATURE MATRIX
# ============================================================================

# Create final dataframe with target
df_final = df_clean.copy()
df_final["survived"] = y_raw

# Encode sex (binary)
df_final["sex"] = df_final["sex"].map({"male": 0, "female": 1})
print("Sex: Binary encoding (male=0, female=1)")

# One-hot encode embarked
embarked_dummies = pd.get_dummies(
    df_final["embarked"], prefix="embarked", drop_first=True, dtype=int
)
df_final = pd.concat([df_final, embarked_dummies], axis=1)
print("Embarked: One-hot encoding (dropped first category)")

# Drop original embarked column
df_final = df_final.drop(columns=["embarked"])

# Verify no missing values
print(f"\nMissing values after imputation: {df_final.isnull().sum().sum()}")

# Create X and y
X = df_final.drop("survived", axis=1)
y = df_final["survived"]

print("\n" + "=" * 50)
print("FINAL FEATURE MATRIX")
print("=" * 50)
print(f"Number of samples (N): {X.shape[0]}")
print(f"Number of features (p): {X.shape[1]}")
print(f"\nFeatures: {list(X.columns)}")
print("\nNote: fare_log is log1p-transformed fare (reduces skewness and makes data normal, which is essential because ml models assume normal data (generally)")

In [ ]:
# Look at the final feature matrix
df_final.head()

In [ ]:
# ============================================================================
# SAVE CLEANED DATA
# ============================================================================

df_final.to_csv("../titanic_cleaned.csv", index=False)
np.save("../X_full.npy", X.values)
np.save("../y_full.npy", y.values)

print("\nSaved cleaned data:")
print("   - titanic_cleaned.csv")
print("   - X_full.npy")
print("   - y_full.npy")

# ============================================================================
# VERIFICATION
# ============================================================================

print("\n" + "=" * 50)
print("VERIFICATION")
print("=" * 50)
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"Class distribution: {np.bincount(y.astype(int))}")
print(f"\nFeature summary")
print(X.describe())

## Correlation Analysis & Pairplots

In [ ]:
# ============================================================================
# CORRELATION MATRIX HEATMAP
# ============================================================================

# Calculate correlation matrix on df_final
correlation_matrix_full = df_final.corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    correlation_matrix_full,
    annot=True,           # Show correlation values
    fmt='.2f',            # 2 decimal places
    cmap='coolwarm',      # Red-blue colormap
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"shrink": 0.8}
)
ax.set_title("Full Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../figures/correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

# ============================================================================
# CORRELATION WITH TARGET - BAR PLOT
# ============================================================================

# Calculate correlation of features with target (excluding target itself)
corr_with_target = correlation_matrix_full["survived"].drop("survived").sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#4ECDC4' if c > 0 else '#FF6B6B' for c in corr_with_target.values]
bars = ax.barh(corr_with_target.index, corr_with_target.values, color=colors, edgecolor='black')
ax.set_xlabel("Correlation with Survival", fontsize=12)
ax.set_title("Feature Correlation with Survival (Target)", fontsize=14, fontweight="bold")
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)

# Add value labels on bars
for bar, val in zip(bars, corr_with_target.values):
    ax.text(
        bar.get_width() + 0.01 if val > 0 else bar.get_width() - 0.05,
        bar.get_y() + bar.get_height()/2,
        f'{val:.2f}',
        va='center',
        fontsize=10
    )

plt.tight_layout()
plt.savefig("../figures/correlation_with_target.png", dpi=300, bbox_inches="tight")
plt.show()

# ============================================================================
# PAIRPLOT
# ============================================================================

# Select a subset of features for pairplot (including target)
features_for_pairplot = ['pclass', 'sex', 'age', 'fare_log', 'survived']

# Create pairplot
fig = sns.pairplot(
    df_final[features_for_pairplot],
    hue='survived',
    palette=['#FF6B6B', '#4ECDC4'],
    diag_kind='kde',
    plot_kws={'alpha': 0.6, 's': 30},
    diag_kws={'alpha': 0.7}
)
fig.fig.suptitle("Pairplot: Feature Relationships by Survival", y=1.02, fontsize=14, fontweight="bold")
plt.savefig("../figures/pairplot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================================
# CORRELATION OBSERVATIONS
# ============================================================================

print("\n" + "=" * 60)
print("CORRELATION OBSERVATIONS")
print("=" * 60)

# Get the actual correlation values
corr_sex = corr_with_target.get('sex', 0)
corr_pclass = corr_with_target.get('pclass', 0)
corr_fare_log = corr_with_target.get('fare_log', 0)
corr_age = corr_with_target.get('age', 0)
corr_parch = corr_with_target.get('parch', 0)
corr_sibsp = corr_with_target.get('sibsp', 0)

# Find strongest positive and strongest negative
strongest_positive = corr_with_target.idxmax()
strongest_negative = corr_with_target.idxmin()

print(f"""
Based on the correlation analysis with survival target:

1. {strongest_positive} ({corr_with_target.max():.2f}) - STRONGEST PREDICTOR
   → Female passengers have significantly higher survival rates

2. {strongest_negative} ({corr_with_target.min():.2f}) - STRONGEST NEGATIVE CORRELATION
   → Higher class number (3rd class) associated with lower survival
   → 1st class passengers had highest survival rates

3. fare_log ({corr_fare_log:.2f}) - {'POSITIVE' if corr_fare_log > 0 else 'NEGATIVE'} CORRELATION
   → Higher fares correlate with higher survival
   → Wealthier passengers had better access to lifeboats

4. age ({corr_age:.2f}) - {'POSITIVE' if corr_age > 0 else 'NEGATIVE'} CORRELATION
   → Older passengers {'more' if corr_age > 0 else 'less'} likely to survive
   → Effect is weaker than other factors

5. parch ({corr_parch:.2f}) and sibsp ({corr_sibsp:.2f}) - WEAK CORRELATIONS
   → Minimal linear relationship with survival
   → May have non-linear effects (e.g., being alone vs large family)
""")

# ============================================================================
# PRINT ALL CORRELATIONS WITH TARGET (dynamic)
# ============================================================================

print("\n" + "-" * 50)
print("Complete feature correlations with survival:")
print("-" * 50)
for feature, corr_value in corr_with_target.items():
    direction = "POSITIVE" if corr_value > 0 else "NEGATIVE"
    print(f"  {feature:12}: {corr_value:.3f} ({direction})")